## Load and Viz Models

In [ ]:
import numpy as np
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
import torch.optim as optim
import cv2
from matplotlib.gridspec import GridSpec
from tqdm import tqdm
import os
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image

In [ ]:
from itertools import product
from scipy.spatial import ConvexHull
import warnings

def find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=1000, min_samples_per_region=5):
    """
    Find linear regions (polytopes) created by a ReLU network on a 2D input plane.
    
    Args:
        model: PyTorch model with ReLU activations (like BaarleNet)
        extent: [xmin, xmax, ymin, ymax] defining the input plane bounds
        grid_resolution: Number of points along each axis for sampling
        min_samples_per_region: Minimum points needed to define a region
    
    Returns:
        List of numpy arrays, each containing corner coordinates of a linear region
    """
    
    def get_relu_pattern(model, x):
        """Get the activation pattern (which ReLUs are active) for input x"""
        pattern = []
        current_x = x
        
        with torch.no_grad():
            for layer in model.layers:
                if isinstance(layer, torch.nn.Linear):
                    current_x = layer(current_x)
                elif isinstance(layer, torch.nn.ReLU):
                    # Record which neurons are active (>0)
                    active = (current_x > 0).float()
                    pattern.append(active)
                    current_x = torch.relu(current_x)
        
        # Flatten and convert to tuple for hashing
        return tuple(torch.cat(pattern, dim=-1).flatten().tolist())
    
    # Create a grid of sample points
    x_vals = np.linspace(extent[0], extent[1], grid_resolution)
    y_vals = np.linspace(extent[2], extent[3], grid_resolution)
    
    # Dictionary to store points by their activation pattern
    pattern_to_points = {}
    
    print(f"Sampling {grid_resolution}x{grid_resolution} points to find activation patterns...")
    
    # Sample points and group by activation pattern
    for i, x in enumerate(x_vals):
        if i % (grid_resolution // 10) == 0:
            print(f"Progress: {i/grid_resolution*100:.1f}%")
            
        for y in y_vals:
            point = torch.tensor([[x, y]], dtype=torch.float32)
            pattern = get_relu_pattern(model, point)
            
            if pattern not in pattern_to_points:
                pattern_to_points[pattern] = []
            pattern_to_points[pattern].append([x, y])
    
    print(f"Found {len(pattern_to_points)} distinct activation patterns")
    
    # Convert regions to convex hulls and find corner points
    linear_regions = []
    
    for pattern, points in pattern_to_points.items():
        if len(points) < min_samples_per_region:
            continue
            
        points_array = np.array(points)
        
        try:
            # Find convex hull of points in this region
            hull = ConvexHull(points_array)
            corners = points_array[hull.vertices]
            linear_regions.append(corners)
            
        except Exception as e:
            warnings.warn(f"Could not compute convex hull for region with {len(points)} points: {e}")
            continue
    
    print(f"Successfully extracted {len(linear_regions)} linear regions")
    
    # Refine boundaries by finding exact hyperplane intersections
    refined_regions = []
    
    for region_corners in linear_regions:
        # For small regions or those touching boundaries, use the sampled corners
        # For larger regions, we could implement exact hyperplane intersection finding
        refined_regions.append(region_corners)
    
    return refined_regions


def visualize_regions(regions, extent=[-1, 1, -1, 1], figsize=(6, 6)):
    """
    Helper function to visualize the found regions (requires matplotlib)
    """
    try:
        import matplotlib.pyplot as plt
        from matplotlib.patches import Polygon
        import matplotlib.colors as mcolors
        
        fig, ax = plt.subplots(1, 1, figsize=figsize)
        
        # Generate colors for each region
        colors = list(mcolors.TABLEAU_COLORS.values())
        
        for i, corners in enumerate(regions):
            color = colors[i % len(colors)]
            
            # Create polygon for this region
            polygon = Polygon(corners, alpha=0.3, facecolor=color, edgecolor='black', linewidth=0.5)
            ax.add_patch(polygon)
            
            # Mark corners
            # ax.scatter(corners[:, 0], corners[:, 1], c='red', s=10, zorder=5, alpha=0.5)
        
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(f'Linear Regions of ReLU Network ({len(regions)} regions)')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')
        
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("matplotlib not available for visualization")

In [ ]:
map=cv2.imread('Baarle-Nassau_-_Baarle-Hertog-en no legend.png')[:,:,(2,1,0)]

belgium_color=np.array([251,234,81])
netherlands_color=np.array([255, 255, 228])

netherlands_region=((map-netherlands_color)**2).sum(-1)<50
belgium_region=((map-belgium_color)**2).sum(-1)<10000

num_points_to_sample=10000 #10000

b_coords=np.array(np.where(belgium_region)).T.astype('float')
n_coords=np.array(np.where(netherlands_region)).T.astype('float')

#flip and normalize
belgium_coords_all=np.zeros_like(b_coords)
netherlands_coords_all=np.zeros_like(n_coords)

belgium_coords_all[:,0]=b_coords[:,1]/(960/2)-1 #Scale to -1 to 1 for now
belgium_coords_all[:,1]=(960-b_coords[:,0])/(960/2)-1
netherlands_coords_all[:,0]=n_coords[:,1]/(960/2)-1 #Scale to -1 to 1 for now
netherlands_coords_all[:,1]=(960-n_coords[:,0])/(960/2)-1

## Hmm to balance to classes or do more area proportional? Not sure yet. 
# belgium_coords=belgium_coords_all[np.random.choice(len(belgium_coords_all), num_points_to_sample), :]
# netherlands_coords=netherlands_coords_all[np.random.choice(len(netherlands_coords_all), num_points_to_sample), :]

X=np.vstack((netherlands_coords_all, belgium_coords_all))
y=np.concatenate((np.zeros(len(netherlands_coords_all)), np.ones(len(belgium_coords_all)))).astype('int')

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

In [ ]:
map.shape

In [ ]:
def viz_descision_boundary(model, res=256):
    plt.clf()
    fig=plt.figure(0,(6,6))
    ax=fig.add_subplot(111)
    
    probe=np.zeros((res,res,2))
    for j, xx in enumerate(np.linspace(-1, 1, res)):
        for k, yy in enumerate(np.linspace(-1, 1, res)):
            probe[j, k]=[yy,xx]
    probe=probe.reshape(res**2, -1)
    with torch.no_grad():
        probe_logits=model(torch.tensor(probe).float())
        probe_logits=probe_logits.detach().numpy().reshape(res,res,2)
        probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))
    
    ax.imshow(map.mean(2), cmap='gray')
    ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
               extent=[0, 960, 960, 0],  # This maps to image coordinates
               alpha=0.7,
               cmap='viridis')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import os
from PIL import Image

def viz_and_save_heatmaps(model, save_dir="../data/hackin/heatmaps", model_name="model", rez=960):
    """
    Generate and save heatmaps for a PyTorch model's output regions.
    
    Args:
        model: PyTorch model that takes 2D input and outputs 2 channels
        save_dir: Directory to save the heatmap images
        model_name: Base name for the saved files
        rez: Resolution for the heatmap generation
    """
    
    # Create save directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Create probe grid at higher resolution
    probe = np.zeros((rez, rez, 2))
    for j, xx in enumerate(np.linspace(-1, 1, rez)):
        for k, yy in enumerate(np.linspace(-1, 1, rez)):
            probe[j, k] = [yy, xx]
    
    probe = probe.reshape(rez**2, -1)
    
    # Get model predictions
    with torch.no_grad():
        probe_logits = model(torch.tensor(probe).float())
        probe_logits = probe_logits.detach().numpy().reshape(rez, rez, 2)
    
    # Get argmax predictions for each pixel
    predictions = np.argmax(probe_logits, axis=2)
    
    # Define colors
    YELLOW = '#ffd35a'
    BLUE = '#65c8d0'
    
    # Create and save heatmaps for each channel
    colors = [BLUE, YELLOW]
    
    for channel in range(2):
        # Create binary mask for this channel
        channel_mask = (predictions == channel).astype(float)
        
        # Create RGBA image
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.set_xlim(0, rez)
        ax.set_ylim(0, rez)
        ax.axis('off')
        
        # Create custom colormap: transparent where mask is 0, solid color where mask is 1
        color_rgb = [int(colors[channel][1:3], 16)/255, 
                     int(colors[channel][3:5], 16)/255, 
                     int(colors[channel][5:7], 16)/255]
        
        # Create binary colormap
        cmap_colors = [(0, 0, 0, 0),  # Transparent for value 0
                       (color_rgb[0], color_rgb[1], color_rgb[2], 1)]  # Solid color for value 1
        cmap = ListedColormap(cmap_colors)
        
        # Plot the binary mask (flip vertically to match image coordinates)
        im = ax.imshow(np.flipud(channel_mask), 
                      extent=[0, rez, 0, rez],
                      cmap=cmap,
                      vmin=0, vmax=1)
        
        # Save as PNG with transparency
        filename = f"{model_name}_{channel}.png"
        filepath = os.path.join(save_dir, filename)
        
        plt.savefig(filepath, 
                   bbox_inches='tight', 
                   pad_inches=0, 
                   transparent=True, 
                   dpi=100)
        
        plt.close()
        
        print(f"Saved {filename} to {save_dir}")
    
    print(f"Heatmaps saved as {model_name}_0.png and {model_name}_1.png")

In [ ]:
X.shape, y.shape, X_tensor.shape, y_tensor.shape

In [ ]:
class BaarleNet(nn.Module):
    def __init__(self, hidden_layers=[64]):
        super(BaarleNet, self).__init__()
        layers = [nn.Linear(2, hidden_layers[0]), nn.ReLU()]
        for i in range(len(hidden_layers)-1):
            layers.append(nn.Linear(hidden_layers[i], hidden_layers[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_layers[-1], 2))
        self.layers=layers
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

## Single Layer

## Shallow [2]

In [ ]:
model_path='../models/2_1.pth'
model = BaarleNet([2])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

## Shallow [3]

In [ ]:
model_path='../models/3_1.pth'
model = BaarleNet([3])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

## Shallow [4]

## Shallow [8]

## Shallow [16]

## Shallow [32]

## Shallow [64]

In [ ]:
model=torch.load('../models/one_layer_64_nuerons_long.pt')

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

## Shallow [128]

## Shallow [256]

## Shallow [512]

In [ ]:
model=torch.load('../models/one_layer_512_nuerons.pt')

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

## Shallow [1024]

In [ ]:
model=torch.load('../models/one_layer_1024_nuerons_long.pt', weights_only=False)

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)

In [ ]:
visualize_regions(regions)

In [ ]:
torch.save(model.state_dict(), '../models/one_layer_1024_nuerons_long.pth')

## Shallow [10k]
- Super ram hungry, let's run on linux - do need to know if these get all the buildings...
- Eh looks like i gotta batch it. 

In [ ]:
X_tensor.shape, y_tensor.shape

In [ ]:
# model=torch.load('../models/one_layer_10k_neurons_long.pt', weights_only=False)

# outputs_batch = model(X_tensor)
# accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

# print(accuracy); viz_descision_boundary(model)

In [ ]:
# Load model
model = torch.load('../models/one_layer_10k_neurons_long.pt', weights_only=False)
model.eval()  # Set to evaluation mode

# Batch processing parameters
batch_size = 1000  # Adjust based on your available RAM
total_samples = len(X_tensor)
correct_predictions = 0

# Process in batches
with torch.no_grad():  # Disable gradient computation to save memory
    for i in range(0, total_samples, batch_size):
        # Get batch
        end_idx = min(i + batch_size, total_samples)
        X_batch = X_tensor[i:end_idx]
        y_batch = y_tensor[i:end_idx]
        
        # Forward pass
        outputs_batch = model(X_batch)
        
        # Calculate predictions and accumulate correct count
        predictions = torch.argmax(outputs_batch, dim=1)
        correct_predictions += (predictions == y_batch).sum().item()

# Calculate final accuracy
accuracy = correct_predictions / total_samples
print(f"Accuracy: {accuracy}")

# Visualize decision boundary (this will still use the full model)
viz_descision_boundary(model)

In [ ]:
torch.save(model.state_dict(), '../models/one_layer_10k_neurons_long.pth')

## Shallow [100k]

In [ ]:
# model=torch.load('../models/one_layer_100k_neurons_long.pt', weights_only=False)

# Load model
model = torch.load('../models/one_layer_100k_neurons_long.pt', weights_only=False)
model.eval()  # Set to evaluation mode
model.to('cpu')

# Batch processing parameters
batch_size = 1000  # Adjust based on your available RAM
total_samples = len(X_tensor)
correct_predictions = 0

# Process in batches
with torch.no_grad():  # Disable gradient computation to save memory
    for i in range(0, total_samples, batch_size):
        # Get batch
        end_idx = min(i + batch_size, total_samples)
        X_batch = X_tensor[i:end_idx]
        y_batch = y_tensor[i:end_idx]
        
        # Forward pass
        outputs_batch = model(X_batch)
        
        # Calculate predictions and accumulate correct count
        predictions = torch.argmax(outputs_batch, dim=1)
        correct_predictions += (predictions == y_batch).sum().item()

# Calculate final accuracy
accuracy = correct_predictions / total_samples
print(f"Accuracy: {accuracy}")

# Visualize decision boundary (this will still use the full model)
viz_descision_boundary(model)

In [ ]:
torch.save(model.state_dict(), '../models/one_layer_100k_neurons_long.pth')

## [2,2]

In [ ]:
model_path='../models/2_2_1.pth'
model = BaarleNet([2,2])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)

In [ ]:
visualize_regions(regions)

## [3,3]

In [ ]:
model_path='../models/3_3_1.pth'
model = BaarleNet([3,3])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

## [8, 8]

In [ ]:
model_path='../models/8_8_1.pth'
model = BaarleNet([8,8])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

In [ ]:
viz_and_save_heatmaps(model, model_name='8_8')

## [16, 16]

In [ ]:
model_path='../models/16_16_1.pth'
model = BaarleNet([16,16])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

## [2,2,2]

In [ ]:
model_path='../models/2_2_2_1.pth'
model = BaarleNet([2,2,2])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

## [16, 16, 16]

In [ ]:
model_path='../models/16_16_16_1.pth'
model = BaarleNet([16,16,16])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=256)
visualize_regions(regions)

Cools.

## 32 32 32

In [ ]:
model_path='../models/32_32_32_1.pth'
model = BaarleNet([32, 32, 32])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=1024);
# visualize_regions(regions)

In [ ]:
visualize_regions(regions)

## 32 32 32 32

In [ ]:
model_path='../models/32_32_32_32_1.pth'
model = BaarleNet([32, 32, 32, 32])
model.load_state_dict(torch.load(model_path))

outputs_batch = model(X_tensor)
accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)

print(accuracy); viz_descision_boundary(model)

In [ ]:
regions = find_linear_regions_empirical(model, extent=[-1, 1, -1, 1], grid_resolution=1024);

In [ ]:
visualize_regions(regions)

- Woah that's legit -> it't intersting that th emodel is able to learn to "focus on the borders" so much, ya know? 
- That might be interesting to try to undersatnd in the training process. 